[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/100.%20The%20Closed-Loop%20%28Reverse%20Logistics%29%20Network%20Design%20Problem/P100-Tier-8_executed.ipynb)

---


# 100. The Closed-Loop Network Design Problem
## Tier 8: A Value-Aligned and Ethical Framework

### Purpose and scope

This notebook demonstrates how an ethical constitution, stakeholder utility functions, a feasibility-repair step, and a simple hill-climbing search can be combined in a small illustrative network-planning example. The input is a dictionary of performance metrics for one candidate plan. The output is a candidate metric dictionary, its stakeholder utilities, a constraint-compliance assessment, and an audit summary.

The example is a framework demonstration rather than a physical closed-loop network design model: it does not represent facilities, product flows, reverse-logistics quantities, or network-design decisions explicitly. Its metrics and thresholds are illustrative and are not legal advice or validated sustainability standards. Compliance checks apply only to the metrics represented in the input. The search uses randomized local changes and does not guarantee a global optimum.

### Modeling assumptions

- Five principles organize the ethical framework: environmental sustainability, social responsibility, economic viability, transparency, and fairness.
- Constraints use an explicit comparison direction. For example, emissions and executive-pay ratio have upper bounds, while wage compliance, profit margin, community investment, and reporting frequency have lower bounds.
- A metric dictionary describes a candidate plan. The stakeholder utility functions reward metrics with positive priority weights; therefore, the environmental metric named `environmental_impact` is interpreted here as a normalized environmental-performance benefit score, where a larger value is better.
- Hard constraints determine feasibility. Soft constraints add a fixed penalty when violated, and guidelines are reported without a penalty.
- The stakeholder objective is the sum of stakeholder utilities. This is a scalar weighted-utility objective, not a Pareto multi-objective optimization model. Stakeholder utilities are not calibrated to a common real-world scale, so their sum is illustrative.
- Constraint `weight` values are retained as descriptive metadata; the objective below does not use them. The search and audit are educational diagnostics, not evidence that a real organization is ethically compliant.

### Method at a glance

The process defines principles and constraints, represents stakeholder preferences, creates an initial metric vector, checks compliance, repairs any hard violations, and searches nearby metric vectors using hill climbing. A neighbor is accepted only when it is hard-feasible and has a strictly larger objective score. The notebook then visualizes the recorded evaluations and produces an audit report.

The framework has five core principles and four stakeholder groups. The constraint list contains seven checks. The optimizer is a local-search demonstration; it does not implement real-time monitoring, learn stakeholder preferences, or prove that a selected plan is ethically optimal.


## Mathematical model and search logic

### Candidate metrics and stakeholder utility

Let a candidate plan be represented by a metric vector $x$, with metric value $x_m$ for metric $m$. Stakeholder $s$ assigns priority weight $w_{sm}$ to metrics they value. The utility function selected for that stakeholder transforms each valued metric before summing its weighted contribution:

$$
U_s(x) = \sum_{m \in P_s} w_{sm} f_s(x_m),
$$

where $P_s$ is the set of metrics prioritized by stakeholder $s$. For a linear utility, $f_s(v)=v$; for a logarithmic utility, $f_s(v)=\log(1+v)$; and for an exponential utility, $f_s(v)=1-\exp(-v)$. The example uses nonnegative metric values, so these transformations are defined. In the code, `Stakeholder.calculate_utility` computes $U_s(x)$ and `EthicalOptimizer.calculate_stakeholder_utility` computes it for every stakeholder.

### Ethical constraints and objective

Each constraint $j$ tests one metric against a threshold $b_j$ using its declared direction. A constraint is violated when the metric fails that comparison. Hard violations make a candidate infeasible; a soft violation incurs half of its configured `violation_penalty`; a guideline violation is recorded without a penalty. The compliance score is a descriptive count-based score, not a probability or a legal certification.

For a candidate that has been repaired to satisfy hard constraints, the search score is

$$
F(x) = \sum_{s \in S} U_s(x) - \sum_{j \in J_{\mathrm{soft}}} p_j(x),
$$

where $S$ is the stakeholder set, $J_{\mathrm{soft}}$ is the set of soft constraints, and $p_j(x)$ is zero when soft constraint $j$ is satisfied and otherwise equals half its configured penalty. Hard constraints are enforced through feasibility checks and repair rather than by relying on a large penalty. The code computes this score inside `EthicalOptimizer.optimize_network`.

### Repair and hill climbing

A hard violation is repaired by setting its metric to the constraint boundary. For an upper-bound constraint, the repair sets the metric to its maximum permitted value; for a lower-bound constraint, it sets the metric to its minimum permitted value. A random neighbor is formed by perturbing one metric. The neighbor replaces the current candidate only if it satisfies every hard constraint and has a strictly higher score.

In pseudocode, with current candidate $x_t$ and a randomly generated neighbor $y_t$:

$$
\begin{aligned}
x_0 &\leftarrow \operatorname{repair}(x_{\mathrm{initial}}), \\
y_t &\leftarrow \operatorname{neighbor}(x_t), \\
x_{t+1} &\leftarrow
\begin{cases}
y_t, & \text{if } y_t \text{ satisfies all hard constraints and } F(y_t)>F(x_t), \\
x_t, & \text{otherwise.}
\end{cases}
\end{aligned}
$$

The search retains the best-scoring candidate encountered and stops after the specified number of iterations. It is a stochastic hill-climbing procedure: accepting only improvements can lead to a local optimum, and the iteration limit is not a proof of convergence to a global optimum. If there are $d$ metrics, $C$ constraints, $S$ stakeholders, and at most $P$ prioritized metrics per stakeholder, one iteration requires roughly $O(C+SP)$ work for candidate evaluation, in addition to copying and perturbing the metric dictionary. Across $T$ iterations, the corresponding evaluation work is roughly $O(T(C+SP))$.


In [ ]:
from collections import defaultdict
from dataclasses import dataclass, field
from enum import Enum
import random
import time
from typing import Any, Dict, List, Optional
import warnings

import matplotlib.pyplot as plt
import numpy as np

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8")

# Fixed seeds make the example's random initialization and neighbor proposals reproducible.
np.random.seed(42)
random.seed(42)

In [ ]:
class EthicalPrinciple(Enum):
    """Principles used to organize the ethical constraints."""

    ENVIRONMENTAL_SUSTAINABILITY = "environmental_sustainability"
    SOCIAL_RESPONSIBILITY = "social_responsibility"
    ECONOMIC_VIABILITY = "economic_viability"
    TRANSPARENCY = "transparency"
    FAIRNESS = "fairness"


@dataclass
class EthicalConstraint:
    """A metric threshold, its comparison direction, and its response to violation."""

    principle: EthicalPrinciple
    constraint_type: str
    description: str
    metric_name: str
    threshold_value: float
    weight: float = 1.0
    violation_penalty: float = 1000.0
    operator: str = "<="

    def __post_init__(self) -> None:
        if self.constraint_type not in {"hard", "soft", "guideline"}:
            raise ValueError("constraint_type must be 'hard', 'soft', or 'guideline'.")
        if self.operator not in {"<=", ">="}:
            raise ValueError("operator must be '<=' or '>='.")


@dataclass
class Stakeholder:
    """A stakeholder group with metric priorities and a utility transformation."""

    stakeholder_id: str
    stakeholder_type: str
    priorities: Dict[str, float] = field(default_factory=dict)
    utility_function: str = "linear"
    risk_tolerance: float = 0.5
    time_horizon: str = "medium"

    def calculate_utility(self, outcomes: Dict[str, float]) -> float:
        """Sum this stakeholder's weighted utilities for metrics they prioritize."""

        utility_functions = {
            "linear": lambda value: value,
            "logarithmic": np.log1p,
            "exponential": lambda value: 1 - np.exp(-value),
        }
        if self.utility_function not in utility_functions:
            raise ValueError(f"Unknown utility function: {self.utility_function}")

        transform = utility_functions[self.utility_function]
        utility = 0.0
        for metric, priority_weight in self.priorities.items():
            if metric in outcomes:
                value = outcomes[metric]
                if self.utility_function == "logarithmic" and value <= -1:
                    raise ValueError("Logarithmic utility requires metric values greater than -1.")
                utility += priority_weight * float(transform(value))
        return utility


@dataclass
class EthicalConstitution:
    """A set of principles, metric constraints, and compliance audit records."""

    constitution_id: str
    core_principles: List[EthicalPrinciple]
    constraints: List[EthicalConstraint] = field(default_factory=list)
    audit_log: List[Dict[str, Any]] = field(default_factory=list)
    compliance_history: List[float] = field(default_factory=list)

    def add_constraint(self, constraint: EthicalConstraint) -> None:
        """Add a threshold check to the constitution."""
        self.constraints.append(constraint)

    def check_compliance(self, solution_metrics: Dict[str, float]) -> Dict[str, Any]:
        """Evaluate all constraints and record the candidate assessment."""

        compliance_result = {
            "is_compliant": True,
            "violations": [],
            "warnings": [],
            "compliance_score": 1.0,
            "penalty_score": 0.0,
        }

        for constraint in self.constraints:
            if constraint.metric_name not in solution_metrics:
                raise KeyError(
                    f"Candidate is missing required metric: {constraint.metric_name}"
                )

            metric_value = solution_metrics[constraint.metric_name]
            if constraint.operator == "<=":
                violated = metric_value > constraint.threshold_value
            else:
                violated = metric_value < constraint.threshold_value

            if not violated:
                continue

            finding = {
                "principle": constraint.principle.value,
                "constraint": constraint.description,
                "metric": constraint.metric_name,
                "value": metric_value,
                "threshold": constraint.threshold_value,
                "operator": constraint.operator,
            }

            if constraint.constraint_type == "hard":
                finding["severity"] = "hard"
                finding["penalty"] = constraint.violation_penalty
                compliance_result["is_compliant"] = False
                compliance_result["violations"].append(finding)
                compliance_result["penalty_score"] += constraint.violation_penalty
            elif constraint.constraint_type == "soft":
                finding["severity"] = "soft"
                finding["penalty"] = constraint.violation_penalty * 0.5
                compliance_result["warnings"].append(finding)
                compliance_result["penalty_score"] += constraint.violation_penalty * 0.5
            else:
                finding["severity"] = "guideline"
                compliance_result["warnings"].append(finding)

        total_constraints = len(self.constraints)
        if total_constraints:
            violation_count = len(compliance_result["violations"])
            warning_count = len(compliance_result["warnings"])
            score = 1.0 - (
                violation_count * 0.5 + warning_count * 0.1
            ) / total_constraints
            compliance_result["compliance_score"] = max(0.0, score)

        # Every assessment is retained so the audit trail includes search evaluations.
        self.audit_log.append(
            {
                "timestamp": time.time(),
                "compliance_result": compliance_result.copy(),
                "solution_metrics": solution_metrics.copy(),
            }
        )
        self.compliance_history.append(compliance_result["compliance_score"])
        return compliance_result


@dataclass
class EthicalOptimizer:
    """A feasibility-repair and hill-climbing search over metric dictionaries."""

    optimizer_id: str
    constitution: EthicalConstitution
    stakeholders: List[Stakeholder] = field(default_factory=list)
    optimization_history: List[Dict[str, Any]] = field(default_factory=list)

    def add_stakeholder(self, stakeholder: Stakeholder) -> None:
        """Include a stakeholder in the summed-utility objective."""
        self.stakeholders.append(stakeholder)

    def calculate_stakeholder_utility(
        self, solution_metrics: Dict[str, float]
    ) -> Dict[str, float]:
        """Calculate the utility of each stakeholder for one candidate."""
        return {
            stakeholder.stakeholder_id: stakeholder.calculate_utility(solution_metrics)
            for stakeholder in self.stakeholders
        }

    def _objective_score(
        self,
        solution_metrics: Dict[str, float],
        compliance_result: Dict[str, Any],
    ) -> tuple[float, Dict[str, float]]:
        """Combine stakeholder utility and the configured soft-constraint penalty."""
        utilities = self.calculate_stakeholder_utility(solution_metrics)
        score = sum(utilities.values()) - compliance_result["penalty_score"]
        return score, utilities

    def optimize_network(
        self,
        initial_solution: Dict[str, float],
        max_iterations: int = 100,
    ) -> Dict[str, Any]:
        """Run the specified repair-and-hill-climbing search."""

        if max_iterations <= 0:
            raise ValueError("max_iterations must be positive.")
        if not initial_solution:
            raise ValueError("initial_solution must contain at least one metric.")

        self.optimization_history = []
        current_solution = initial_solution.copy()
        current_compliance = self.constitution.check_compliance(current_solution)
        if not current_compliance["is_compliant"]:
            current_solution = self._repair_solution(
                current_solution, current_compliance["violations"]
            )
            current_compliance = self.constitution.check_compliance(current_solution)

        best_solution = current_solution.copy()
        best_compliance = current_compliance
        best_score, _ = self._objective_score(best_solution, best_compliance)

        for iteration in range(max_iterations):
            current_score, _ = self._objective_score(
                current_solution, current_compliance
            )
            neighbor_solution = self._generate_neighbor(current_solution)
            neighbor_compliance = self.constitution.check_compliance(neighbor_solution)

            if neighbor_compliance["is_compliant"]:
                neighbor_score, _ = self._objective_score(
                    neighbor_solution, neighbor_compliance
                )
                if neighbor_score > current_score:
                    current_solution = neighbor_solution
                    current_compliance = neighbor_compliance
                    current_score = neighbor_score

            if current_score > best_score:
                best_solution = current_solution.copy()
                best_compliance = current_compliance
                best_score = current_score

            current_utilities = self.calculate_stakeholder_utility(current_solution)
            self.optimization_history.append(
                {
                    "iteration": iteration,
                    "solution": current_solution.copy(),
                    "utilities": current_utilities,
                    "compliance": current_compliance,
                    "objective_score": current_score,
                }
            )

        final_compliance = self.constitution.check_compliance(best_solution)
        final_utilities = self.calculate_stakeholder_utility(best_solution)
        final_score, _ = self._objective_score(best_solution, final_compliance)

        return {
            "best_solution": best_solution,
            "best_score": final_score,
            "final_compliance": final_compliance,
            "final_utilities": final_utilities,
            "total_iterations": max_iterations,
            "convergence_achieved": final_compliance["is_compliant"],
        }

    def _repair_solution(
        self,
        solution: Dict[str, float],
        violations: List[Dict[str, Any]],
    ) -> Dict[str, float]:
        """Move each hard-violating metric to its declared feasible boundary."""
        repaired_solution = solution.copy()
        for violation in violations:
            metric = violation["metric"]
            repaired_solution[metric] = violation["threshold"]
        return repaired_solution

    def _generate_neighbor(self, solution: Dict[str, float]) -> Dict[str, float]:
        """Perturb one metric to create a nearby candidate for local search."""
        neighbor = solution.copy()
        metric_key = random.choice(list(neighbor))
        current_value = neighbor[metric_key]
        perturbation = np.random.normal(0, abs(current_value) * 0.1)
        neighbor[metric_key] = max(0.0, current_value + perturbation)
        return neighbor

    def generate_audit_report(self) -> Dict[str, Any]:
        """Summarize recorded compliance evaluations and the final search iteration."""
        if not self.optimization_history:
            return {"error": "No optimization history is available."}

        compliance_scores = self.constitution.compliance_history
        if compliance_scores:
            average_compliance = float(np.mean(compliance_scores))
            minimum_compliance = float(np.min(compliance_scores))
            maximum_compliance = float(np.max(compliance_scores))
            compliance_trend = (
                "improving"
                if compliance_scores[-1] > compliance_scores[0]
                else "stable or declining"
            )
        else:
            average_compliance = minimum_compliance = maximum_compliance = 0.0
            compliance_trend = "unavailable"

        principle_violations: Dict[str, int] = defaultdict(int)
        principle_warnings: Dict[str, int] = defaultdict(int)
        for audit_entry in self.constitution.audit_log:
            result = audit_entry["compliance_result"]
            for violation in result["violations"]:
                principle_violations[violation["principle"]] += 1
            for warning in result["warnings"]:
                principle_warnings[warning["principle"]] += 1

        final_utilities = self.optimization_history[-1]["utilities"]
        utility_values = list(final_utilities.values())
        if utility_values:
            average_utility = float(np.mean(utility_values))
            utility_std = float(np.std(utility_values))
            utility_range = float(max(utility_values) - min(utility_values))
            fairness_ratio = utility_std / max(abs(average_utility), 1e-12)
            utility_fairness = float(np.clip(1.0 - fairness_ratio, 0.0, 1.0))
        else:
            average_utility = utility_std = utility_range = utility_fairness = 0.0

        utility_statistics = {
            "total_utility": float(sum(utility_values)),
            "average_utility": average_utility,
            "utility_std": utility_std,
            "utility_range": utility_range,
        }

        return {
            "audit_timestamp": time.time(),
            "constitution_id": self.constitution.constitution_id,
            "total_optimizations": len(self.optimization_history),
            "compliance_statistics": {
                "average_compliance": average_compliance,
                "min_compliance": minimum_compliance,
                "max_compliance": maximum_compliance,
                "compliance_trend": compliance_trend,
                "recorded_evaluations": len(compliance_scores),
            },
            "violation_analysis": {
                "total_violations": sum(principle_violations.values()),
                "total_warnings": sum(principle_warnings.values()),
                "violations_by_principle": dict(principle_violations),
                "warnings_by_principle": dict(principle_warnings),
            },
            "stakeholder_analysis": {
                "number_of_stakeholders": len(self.stakeholders),
                "utility_statistics": utility_statistics,
                "stakeholder_utilities": final_utilities,
                "utility_fairness": utility_fairness,
            },
            "ethical_performance": {
                "overall_score": average_compliance * utility_fairness,
                "recommendations": self._generate_recommendations(
                    principle_violations, principle_warnings
                ),
            },
        }

    def _generate_recommendations(
        self,
        violations: Dict[str, int],
        warnings: Dict[str, int],
    ) -> List[str]:
        """Create simple audit prompts from the recorded finding counts."""
        recommendations = []
        if violations:
            most_violated = max(violations, key=violations.get)
            recommendations.append(
                f"Review {most_violated} findings ({violations[most_violated]} occurrences)."
            )
        if warnings:
            most_warned = max(warnings, key=warnings.get)
            recommendations.append(
                f"Review {most_warned} warnings ({warnings[most_warned]} occurrences)."
            )
        if sum(violations.values()) > 5:
            recommendations.append("Review hard-constraint definitions and monitoring procedures.")
        if sum(warnings.values()) > 10:
            recommendations.append("Review soft-constraint thresholds and stakeholder priorities.")
        recommendations.append(
            "Continue periodic audits and consultation with affected stakeholders."
        )
        return recommendations

## 1. Define the constitution and stakeholders

The constitution translates principles into measurable checks. Comparison direction matters: an upper bound is appropriate for emissions, waste, and executive-pay ratio, while minimum wage compliance, profit margin, community investment, and reporting frequency use lower bounds. Stakeholder priorities describe which metrics enter each utility calculation; a priority of zero or an omitted metric means that stakeholder receives no utility contribution from that metric.


In [ ]:
print("Initializing the value-aligned ethical framework...")
print("=" * 55)

constitution = EthicalConstitution(
    constitution_id="ethics_001",
    core_principles=[
        EthicalPrinciple.ENVIRONMENTAL_SUSTAINABILITY,
        EthicalPrinciple.SOCIAL_RESPONSIBILITY,
        EthicalPrinciple.ECONOMIC_VIABILITY,
        EthicalPrinciple.TRANSPARENCY,
        EthicalPrinciple.FAIRNESS,
    ],
)

constraints = [
    EthicalConstraint(
        principle=EthicalPrinciple.ENVIRONMENTAL_SUSTAINABILITY,
        constraint_type="hard",
        description="Carbon emissions must not exceed the illustrative limit.",
        metric_name="carbon_emissions",
        threshold_value=100.0,
        weight=2.0,
        violation_penalty=5000.0,
        operator="<=",
    ),
    EthicalConstraint(
        principle=EthicalPrinciple.ENVIRONMENTAL_SUSTAINABILITY,
        constraint_type="soft",
        description="Waste generation should remain within the illustrative target.",
        metric_name="waste_generated",
        threshold_value=50.0,
        weight=1.5,
        violation_penalty=2000.0,
        operator="<=",
    ),
    EthicalConstraint(
        principle=EthicalPrinciple.SOCIAL_RESPONSIBILITY,
        constraint_type="hard",
        description="Minimum-wage compliance must meet the illustrative minimum.",
        metric_name="minimum_wage_compliance",
        threshold_value=0.95,
        weight=2.5,
        violation_penalty=3000.0,
        operator=">=",
    ),
    EthicalConstraint(
        principle=EthicalPrinciple.SOCIAL_RESPONSIBILITY,
        constraint_type="soft",
        description="Community investment should meet the illustrative target.",
        metric_name="community_investment",
        threshold_value=5.0,
        weight=1.0,
        violation_penalty=1000.0,
        operator=">=",
    ),
    EthicalConstraint(
        principle=EthicalPrinciple.ECONOMIC_VIABILITY,
        constraint_type="hard",
        description="Profit margin must meet the illustrative minimum.",
        metric_name="profit_margin",
        threshold_value=0.05,
        weight=2.0,
        violation_penalty=4000.0,
        operator=">=",
    ),
    EthicalConstraint(
        principle=EthicalPrinciple.TRANSPARENCY,
        constraint_type="guideline",
        description="Reporting frequency should meet the illustrative target.",
        metric_name="reporting_frequency",
        threshold_value=12.0,
        weight=1.0,
        violation_penalty=500.0,
        operator=">=",
    ),
    EthicalConstraint(
        principle=EthicalPrinciple.FAIRNESS,
        constraint_type="soft",
        description="Executive-pay ratio should remain within the illustrative target.",
        metric_name="executive_pay_ratio",
        threshold_value=20.0,
        weight=1.5,
        violation_penalty=1500.0,
        operator="<=",
    ),
]

for constraint in constraints:
    constitution.add_constraint(constraint)

print(f"Constitution: {constitution.constitution_id}")
print(f"Core principles: {[principle.value for principle in constitution.core_principles]}")
print(f"Ethical constraints: {len(constitution.constraints)}")

stakeholders = [
    Stakeholder(
        stakeholder_id="customers",
        stakeholder_type="customer",
        priorities={
            "service_quality": 0.4,
            "price_affordability": 0.3,
            "product_availability": 0.3,
        },
        utility_function="linear",
        risk_tolerance=0.3,
        time_horizon="short",
    ),
    Stakeholder(
        stakeholder_id="employees",
        stakeholder_type="employee",
        priorities={
            "wage_level": 0.3,
            "job_security": 0.3,
            "working_conditions": 0.2,
            "career_development": 0.2,
        },
        utility_function="logarithmic",
        risk_tolerance=0.4,
        time_horizon="medium",
    ),
    Stakeholder(
        stakeholder_id="community",
        stakeholder_type="community",
        priorities={
            "environmental_impact": 0.4,
            "local_employment": 0.3,
            "community_investment": 0.3,
        },
        utility_function="exponential",
        risk_tolerance=0.6,
        time_horizon="long",
    ),
    Stakeholder(
        stakeholder_id="investors",
        stakeholder_type="investor",
        priorities={
            "profit_margin": 0.5,
            "market_share": 0.3,
            "growth_rate": 0.2,
        },
        utility_function="linear",
        risk_tolerance=0.7,
        time_horizon="medium",
    ),
]

optimizer = EthicalOptimizer(
    optimizer_id="eth_opt_001",
    constitution=constitution,
)
for stakeholder in stakeholders:
    optimizer.add_stakeholder(stakeholder)

print(f"\nStakeholders added: {len(optimizer.stakeholders)}")
for stakeholder in optimizer.stakeholders:
    print(f"  {stakeholder.stakeholder_id.title()}: {stakeholder.stakeholder_type}")
    print(f"    Prioritized metrics: {list(stakeholder.priorities)}")

## 2. Create and assess an initial candidate

The candidate is a dictionary whose keys must include every metric referenced by the constitution. Its values are illustrative performance measures, not physical network flows. The first compliance assessment identifies hard violations for repair, soft warnings for penalization, and guideline warnings for reporting. Stakeholder utilities are calculated independently so their contributions can be inspected before the search.


In [ ]:
def generate_initial_solution() -> Dict[str, float]:
    """Generate the example's initial metric dictionary using the fixed seed."""
    return {
        # Economic metrics
        "profit_margin": float(np.random.uniform(0.02, 0.15)),
        "market_share": float(np.random.uniform(0.1, 0.4)),
        "growth_rate": float(np.random.uniform(0.02, 0.12)),
        # Customer service metrics
        "service_quality": float(np.random.uniform(0.7, 0.95)),
        "price_affordability": float(np.random.uniform(0.6, 0.9)),
        "product_availability": float(np.random.uniform(0.8, 0.98)),
        # Employee metrics
        "wage_level": float(np.random.uniform(0.8, 1.2)),
        "job_security": float(np.random.uniform(0.7, 0.95)),
        "working_conditions": float(np.random.uniform(0.75, 0.92)),
        "career_development": float(np.random.uniform(0.6, 0.88)),
        # Community metrics
        "environmental_impact": float(np.random.uniform(0.5, 1.5)),
        "local_employment": float(np.random.uniform(0.3, 0.8)),
        "community_investment": float(np.random.uniform(1.0, 8.0)),
        # Metrics used by ethical constraints
        "carbon_emissions": float(np.random.uniform(50, 150)),
        "waste_generated": float(np.random.uniform(20, 80)),
        "minimum_wage_compliance": float(np.random.uniform(0.85, 1.0)),
        "executive_pay_ratio": float(np.random.uniform(10, 35)),
        "reporting_frequency": float(np.random.uniform(8, 16)),
    }


initial_solution = generate_initial_solution()
print("Initial candidate metrics:")
print("=" * 35)
for metric, value in initial_solution.items():
    print(f"{metric}: {value:.3f}")

initial_compliance = constitution.check_compliance(initial_solution)
print("\nInitial compliance assessment:")
print(f"Hard-feasible: {initial_compliance['is_compliant']}")
print(f"Compliance score: {initial_compliance['compliance_score']:.3f}")
print(f"Hard violations: {len(initial_compliance['violations'])}")
print(f"Warnings: {len(initial_compliance['warnings'])}")
print(f"Penalty score: {initial_compliance['penalty_score']:.2f}")

if initial_compliance["violations"]:
    print("\nHard-constraint findings:")
    for violation in initial_compliance["violations"]:
        print(
            f"  - {violation['principle']}: {violation['metric']} "
            f"({violation['value']:.3f} {violation['operator']} "
            f"{violation['threshold']:.3f} is false)"
        )

initial_utilities = optimizer.calculate_stakeholder_utility(initial_solution)
initial_total_utility = sum(initial_utilities.values())
print("\nInitial stakeholder utilities:")
for stakeholder_id, utility in initial_utilities.items():
    print(f"  {stakeholder_id.title()}: {utility:.3f}")
print(f"Total stakeholder utility: {initial_total_utility:.3f}")

## 3. Run the repair-and-hill-climbing search

The optimizer first repairs hard violations in the initial candidate. At each iteration it perturbs one metric, evaluates the neighbor, and accepts it only when it is hard-feasible and improves the penalized summed-utility score. The retained best candidate is therefore feasible with respect to the hard constraints, provided the repair can satisfy the declared thresholds. Soft penalties can influence the search, while guideline findings do not affect the score.

The Boolean field `convergence_achieved` means only that the returned candidate satisfies the hard constraints. It does not certify that the search has reached a local or global optimum.


In [ ]:
print("Running ethical metric search...")
print("=" * 45)

optimization_result = optimizer.optimize_network(
    initial_solution=initial_solution,
    max_iterations=50,
)

print("\nSearch results:")
print(f"Best objective score: {optimization_result['best_score']:.3f}")
print(f"Hard-feasible candidate found: {optimization_result['convergence_achieved']}")
print(f"Iterations: {optimization_result['total_iterations']}")

final_compliance = optimization_result["final_compliance"]
final_utilities = optimization_result["final_utilities"]
final_total_utility = sum(final_utilities.values())

print("\nReturned candidate compliance:")
print(f"Hard-feasible: {final_compliance['is_compliant']}")
print(f"Compliance score: {final_compliance['compliance_score']:.3f}")
print(f"Hard violations: {len(final_compliance['violations'])}")
print(f"Warnings: {len(final_compliance['warnings'])}")
print(f"Penalty score: {final_compliance['penalty_score']:.2f}")

print("\nReturned candidate stakeholder utilities:")
for stakeholder_id, utility in final_utilities.items():
    print(f"  {stakeholder_id.title()}: {utility:.3f}")
print(f"Total stakeholder utility: {final_total_utility:.3f}")

utility_improvement = final_total_utility - initial_total_utility
compliance_improvement = (
    final_compliance["compliance_score"] - initial_compliance["compliance_score"]
)
print("\nInitial-to-returned-candidate comparisons:")
print(f"Stakeholder utility difference: {utility_improvement:+.3f}")
print(f"Compliance-score difference: {compliance_improvement:+.3f}")

print("\nSelected metric changes:")
key_metrics = [
    "carbon_emissions",
    "minimum_wage_compliance",
    "profit_margin",
    "executive_pay_ratio",
]
for metric in key_metrics:
    initial_value = initial_solution[metric]
    final_value = optimization_result["best_solution"][metric]
    print(
        f"  {metric}: {initial_value:.3f} -> {final_value:.3f} "
        f"({final_value - initial_value:+.3f})"
    )

assert final_compliance["is_compliant"]
assert set(final_utilities) == {stakeholder.stakeholder_id for stakeholder in stakeholders}

## 4. Visualize the search and metric profile

The compliance-history panel records every compliance evaluation, including initial, neighbor, repair, and final checks; it is not a one-point-per-iteration convergence curve. The stakeholder panel compares utilities for the initial and returned candidates. The principle panel counts findings across recorded evaluations, so repeated checks of the same candidate can contribute more than once. The radar chart uses explicit illustrative normalizations and should be read as a profile, not as a unit-consistent comparison across metrics.


In [ ]:
def visualize_ethical_optimization(
    optimizer: EthicalOptimizer,
    optimization_result: Dict[str, Any],
    initial_solution: Dict[str, float],
) -> None:
    """Plot compliance evaluations, stakeholder utilities, findings, and metric scores."""

    fig = plt.figure(figsize=(16, 12))
    ax1 = fig.add_subplot(2, 2, 1)
    ax2 = fig.add_subplot(2, 2, 2)
    ax3 = fig.add_subplot(2, 2, 3)
    ax4 = fig.add_subplot(2, 2, 4, projection="polar")

    compliance_scores = optimizer.constitution.compliance_history
    ax1.plot(
        range(1, len(compliance_scores) + 1),
        compliance_scores,
        color="tab:blue",
        linewidth=1.5,
    )
    ax1.axhline(1.0, color="tab:green", linestyle="--", label="Score of 1.0")
    ax1.set_title("Compliance Score by Recorded Evaluation")
    ax1.set_xlabel("Recorded evaluation")
    ax1.set_ylabel("Count-based compliance score")
    ax1.set_ylim(0, 1.05)
    ax1.grid(True, alpha=0.3)
    ax1.legend()

    initial_utilities = optimizer.calculate_stakeholder_utility(initial_solution)
    final_utilities = optimization_result["final_utilities"]
    stakeholder_ids = list(initial_utilities)
    positions = np.arange(len(stakeholder_ids))
    width = 0.35
    initial_values = [initial_utilities[key] for key in stakeholder_ids]
    final_values = [final_utilities[key] for key in stakeholder_ids]
    ax2.bar(
        positions - width / 2,
        initial_values,
        width,
        label="Initial candidate",
        color="lightcoral",
    )
    ax2.bar(
        positions + width / 2,
        final_values,
        width,
        label="Returned candidate",
        color="lightblue",
    )
    ax2.set_title("Stakeholder Utility Comparison")
    ax2.set_xlabel("Stakeholder")
    ax2.set_ylabel("Utility (illustrative scale)")
    ax2.set_xticks(positions)
    ax2.set_xticklabels([key.title() for key in stakeholder_ids])
    ax2.grid(True, axis="y", alpha=0.3)
    ax2.legend()

    principle_violations: Dict[str, int] = defaultdict(int)
    principle_warnings: Dict[str, int] = defaultdict(int)
    for audit_entry in optimizer.constitution.audit_log:
        result = audit_entry["compliance_result"]
        for finding in result["violations"]:
            principle_violations[finding["principle"]] += 1
        for finding in result["warnings"]:
            principle_warnings[finding["principle"]] += 1

    principles = list(EthicalPrinciple)
    principle_names = [principle.value.replace("_", " ").title() for principle in principles]
    violation_counts = [
        principle_violations.get(principle.value, 0) for principle in principles
    ]
    warning_counts = [
        principle_warnings.get(principle.value, 0) for principle in principles
    ]
    positions = np.arange(len(principles))
    ax3.bar(
        positions - width / 2,
        violation_counts,
        width,
        label="Hard violations",
        color="tab:red",
    )
    ax3.bar(
        positions + width / 2,
        warning_counts,
        width,
        label="Warnings",
        color="tab:orange",
    )
    ax3.set_title("Recorded Findings by Ethical Principle")
    ax3.set_xlabel("Ethical principle")
    ax3.set_ylabel("Finding count")
    ax3.set_xticks(positions)
    ax3.set_xticklabels(principle_names, rotation=35, ha="right")
    ax3.grid(True, axis="y", alpha=0.3)
    ax3.legend()

    key_metrics = [
        "carbon_emissions",
        "minimum_wage_compliance",
        "profit_margin",
        "executive_pay_ratio",
        "community_investment",
    ]

    def normalize_metric(metric: str, value: float) -> float:
        """Map selected metric values to illustrative 0-to-1 desirability scores."""
        if metric == "carbon_emissions":
            return float(np.clip(1.0 - value / 100.0, 0.0, 1.0))
        if metric == "minimum_wage_compliance":
            return float(np.clip(value, 0.0, 1.0))
        if metric == "profit_margin":
            return float(np.clip(value / 0.2, 0.0, 1.0))
        if metric == "executive_pay_ratio":
            return float(np.clip(1.0 - value / 50.0, 0.0, 1.0))
        if metric == "community_investment":
            return float(np.clip(value / 10.0, 0.0, 1.0))
        raise KeyError(f"No normalization is defined for {metric}.")

    initial_values = [
        normalize_metric(metric, initial_solution[metric]) for metric in key_metrics
    ]
    final_solution = optimization_result["best_solution"]
    final_values = [
        normalize_metric(metric, final_solution[metric]) for metric in key_metrics
    ]
    angles = np.linspace(0, 2 * np.pi, len(key_metrics), endpoint=False)
    closed_angles = np.append(angles, angles[0])
    initial_profile = np.append(initial_values, initial_values[0])
    final_profile = np.append(final_values, final_values[0])
    ax4.plot(closed_angles, initial_profile, marker="o", label="Initial candidate")
    ax4.plot(closed_angles, final_profile, marker="o", label="Returned candidate")
    ax4.fill(closed_angles, final_profile, alpha=0.2)
    ax4.set_xticks(angles)
    ax4.set_xticklabels([metric.replace("_", " ").title() for metric in key_metrics])
    ax4.set_ylim(0, 1)
    ax4.set_title("Illustrative Normalized Metric Profile", pad=20)
    ax4.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15))

    fig.tight_layout()
    plt.show()


visualize_ethical_optimization(
    optimizer=optimizer,
    optimization_result=optimization_result,
    initial_solution=initial_solution,
)

## 5. Review the audit report

The audit report summarizes the compliance checks recorded during this run and the stakeholder utilities from the final recorded search iteration. Its overall score combines the mean count-based compliance score with a clipped utility-dispersion diagnostic. Neither this score nor the utility-fairness statistic is a validated ethical standard: stakeholders' utility scales can differ, and a low dispersion in utility does not by itself establish fairness. The report is useful for tracing the example's calculations and identifying questions for further review.


In [ ]:
audit_report = optimizer.generate_audit_report()
if "error" in audit_report:
    raise RuntimeError(audit_report["error"])

print("\n" + "=" * 58)
print("ETHICAL FRAMEWORK AUDIT REPORT")
print("=" * 58)
print(f"\nConstitution: {audit_report['constitution_id']}")
print(
    "Audit time: "
    + time.strftime(
        "%Y-%m-%d %H:%M:%S",
        time.localtime(audit_report["audit_timestamp"]),
    )
)
print(f"Search iterations recorded: {audit_report['total_optimizations']}")

compliance_stats = audit_report["compliance_statistics"]
print("\nCompliance evaluation summary:")
print(f"  Recorded evaluations: {compliance_stats['recorded_evaluations']}")
print(f"  Mean compliance score: {compliance_stats['average_compliance']:.3f}")
print(f"  Minimum compliance score: {compliance_stats['min_compliance']:.3f}")
print(f"  Maximum compliance score: {compliance_stats['max_compliance']:.3f}")
print(f"  First-to-last assessment: {compliance_stats['compliance_trend']}")

violation_analysis = audit_report["violation_analysis"]
print("\nRecorded findings:")
print(f"  Hard-violation findings: {violation_analysis['total_violations']}")
print(f"  Warning findings: {violation_analysis['total_warnings']}")
if violation_analysis["violations_by_principle"]:
    print("  Hard violations by principle:")
    for principle, count in violation_analysis["violations_by_principle"].items():
        print(f"    - {principle.replace('_', ' ').title()}: {count}")
if violation_analysis["warnings_by_principle"]:
    print("  Warnings by principle:")
    for principle, count in violation_analysis["warnings_by_principle"].items():
        print(f"    - {principle.replace('_', ' ').title()}: {count}")

stakeholder_analysis = audit_report["stakeholder_analysis"]
utility_stats = stakeholder_analysis["utility_statistics"]
print("\nStakeholder summary:")
print(f"  Stakeholder groups: {stakeholder_analysis['number_of_stakeholders']}")
print(f"  Total utility: {utility_stats['total_utility']:.3f}")
print(f"  Mean utility: {utility_stats['average_utility']:.3f}")
print(f"  Utility standard deviation: {utility_stats['utility_std']:.3f}")
print(f"  Utility range: {utility_stats['utility_range']:.3f}")
print(f"  Clipped utility-dispersion diagnostic: {stakeholder_analysis['utility_fairness']:.3f}")
for stakeholder_id, utility in stakeholder_analysis["stakeholder_utilities"].items():
    print(f"  {stakeholder_id.title()} utility: {utility:.3f}")

ethical_performance = audit_report["ethical_performance"]
print("\nIllustrative combined diagnostic:")
print(f"  Overall score: {ethical_performance['overall_score']:.3f}")
print("  Audit prompts:")
for index, recommendation in enumerate(ethical_performance["recommendations"], 1):
    print(f"    {index}. {recommendation}")

overall_score = ethical_performance["overall_score"]
if overall_score > 0.8:
    status = "High illustrative diagnostic score"
elif overall_score > 0.6:
    status = "Moderate-to-high illustrative diagnostic score"
elif overall_score > 0.4:
    status = "Moderate illustrative diagnostic score"
else:
    status = "Low illustrative diagnostic score"
print(f"\nDiagnostic category: {status}")

print("\nInterpretation checks:")
if final_compliance["is_compliant"]:
    print("  The returned candidate satisfies the modeled hard constraints.")
else:
    print("  The returned candidate has at least one modeled hard-constraint violation.")
if violation_analysis["total_violations"] == 0:
    print("  No hard-violation findings were recorded in the audit log.")
print("  The diagnostic score is not a certification of ethical performance.")

## 6. Interpretation, strengths, and limitations

### Why this framework is useful

- **Explicit governance:** principles are connected to named metrics and constraint directions, making the modeled rules inspectable.
- **Stakeholder representation:** distinct priority dictionaries and utility transformations expose how groups value the modeled outcomes.
- **Feasibility handling:** hard constraints are checked and repaired before candidates are retained, while soft constraints and guidelines have distinct roles.
- **Traceability:** candidate metrics and compliance findings are recorded so that the example's search assessments can be reviewed.
- **Interpretability:** the score is built from stakeholder utilities and explicit soft-constraint penalties.

### Limitations

- The metric vector is not a network-design formulation: there are no facility, flow, capacity, reverse-logistics, or material-balance decisions.
- The sum of stakeholder utilities is sensitive to metric scales, transformations, and priority weights. The illustrative values do not establish comparable real-world welfare units.
- The compliance score counts findings rather than measuring the magnitude or social consequences of violations. Hard feasibility and the score should therefore be read separately.
- The local search uses random one-metric perturbations and accepts strict improvements. It can stop at a local optimum, and its result depends on initialization and random proposals.
- A repair places a violated metric exactly at its threshold. Real applications may require uncertainty margins, measurement error models, and domain-specific repair rules.
- Audit timestamps and repeated candidate evaluations support traceability in this demonstration; they do not provide independent oversight, ongoing production monitoring, or accountability by themselves.
- Principles, thresholds, metric definitions, and stakeholder weights require legitimate stakeholder and domain review before any operational use.

### When the approach may be informative

This pattern is useful as a teaching example when a decision process needs explicit stakeholder utilities, directional metric constraints, local search, and an inspectable audit record. A larger application would require a domain-grounded network model, validated data and thresholds, careful utility calibration, and governance procedures appropriate to the affected stakeholders. Those extensions are outside the scope of this notebook.

### Key takeaways

1. A constraint must state both its threshold and whether the metric is bounded above or below.
2. Hard feasibility, soft penalties, and guideline reporting are different modeling roles and should not be conflated.
3. Summed stakeholder utility is a scalarization choice; it does not automatically ensure fairness or resolve value conflicts.
4. Hill climbing is a local-search procedure, so an improved candidate is not necessarily globally optimal.
5. An audit trail makes modeled checks reviewable, but ethical legitimacy depends on sound measures, transparent governance, and meaningful stakeholder participation.
